In [1]:
import numpy as np
import cvxpy as cp
import flexitroid.utils.population_generator as pg
import flexitroid.aggregations.v1g_aggregator as agg
from importlib import reload
reload(agg)

<module 'flexitroid.aggregations.v1g_aggregator' from '/Users/karkhi/Documents/repos/flextroid/flexitroid/aggregations/v1g_aggregator.py'>

In [2]:
def solve_lp(pop_agg, a_l, a_u, c):
    T = pop_agg.T
    n_dev = len(pop_agg.devices)
    UI = [cp.Variable(T) for _ in range(n_dev)]
    u_agg = cp.Variable(T)

    objective = cp.Minimize(c.T@u_agg)
    constraints = [u_agg == cp.sum(UI),  a_l <= np.eye(T)@u_agg, np.eye(T)@u_agg <= a_u]

    for i in range(n_dev):
        A, b = pop_agg.devices[i].A_b()
        constraints.append(A@UI[i] <= b)
    prob = cp.Problem(objective, constraints)
    prob.solve()
    return u_agg.value

In [3]:
T = 4

pop = pg.PopulationGenerator(T, v1g_count=1000)

a_l = np.zeros(T)
a_u = 900 * np.ones(T)
pop_agg = agg.V1GConstrainted(pop.device_list, a_l, a_u)

c = np.random.uniform(0,1, T)


In [4]:
solve_lp(pop_agg, a_l, a_u, c)

array([ 88.00035171,  29.17738067, 185.92704523,   8.86227582])

In [5]:
pop_agg.major

{(0, 4): array([49.00413921, 38.13926921, 22.48126555,  9.60446427]),
 (2, 4): array([124.00548104,  37.13504305]),
 (0, 2): array([46.50129528, 16.39783477]),
 (1, 3): array([73.75751756, 25.86828251]),
 (1, 4): array([69.86281546, 46.36399565, 17.03898649]),
 (0, 3): array([40.90187737, 21.00739128,  6.64722461])}

In [ ]:
def v_func_pop(j,k,i):
    

In [6]:
import networkx as nx
import numpy as np
import itertools # Only needed for direct calculation if used

# Ensure networkx and potentially scipy are installed
# pip install networkx scipy

def solve_submodular_min_intervals(T_elements, A_set, v_func, a_underline_func, a_overline_func):
    """
    Computes the minimum of f(X) = b(X) - a_underline(X \\ A) + a_overline(A \\ X)
    where b(X) = sum_{j<k} sum_{i=1}^{|X intersect [j..k]|} v_jk(i),
    using a min-cut reduction.

    Args:
        T_elements: A list or sequence of unique, hashable elements in the
                    ground set T (e.g., range(n) or ['a', 'b', 'c']).
                    Assumes an implicit order corresponds to list order for intervals.
        A_set: A set containing elements from T_elements that are in the fixed set A.
        v_func: A function v_func(j, k, i) returning the value v_jk(i) >= 0.
                'j' and 'k' are elements from T_elements defining the interval.
                'i' is a 1-based index (i >= 1).
                It MUST satisfy v_jk(i) >= v_jk(i+1) >= 0 (concavity).
                Should handle i exceeding interval size gracefully (e.g., return 0).
        a_underline_func: Function a_underline_func(element) returning underline{a}(element) >= 0.
        a_overline_func: Function a_overline_func(element) returning overline{a}(element) >= 0.

    Returns:
        min_value: The minimum value found for f(X).
        X_min_set: The set X (subset of T_elements) that achieves the minimum value.

    Raises:
        ValueError: If input constraints (like non-negativity of v, a values) are violated.
        ImportError: If networkx or its backend (like scipy) is not installed.
    """
    try:
        import networkx as nx
    except ImportError:
        raise ImportError("Please install networkx: pip install networkx")

    if not callable(v_func) or not callable(a_underline_func) or not callable(a_overline_func):
         raise TypeError("v_func, a_underline_func, a_overline_func must be callable functions.")

    n = len(T_elements)
    element_list = list(T_elements) # Ensure fixed order
    element_to_idx = {el: i for i, el in enumerate(element_list)}

    # --- 1. Calculate weights w(i) for the modular part g(X) = b(X) + m(X) ---
    # m(X) = sum_{i in X} w(i)
    # w(i) = -a_overline(i) if i in A
    # w(i) = -a_underline(i) if i not in A
    w = np.zeros(n)
    for idx, element in enumerate(element_list):
        if element in A_set:
            cost = a_overline_func(element)
            if cost < 0: raise ValueError(f"a_overline({element}) = {cost} must be non-negative.")
            w[idx] = -cost
        else:
            cost = a_underline_func(element)
            if cost < 0: raise ValueError(f"a_underline({element}) = {cost} must be non-negative.")
            w[idx] = -cost

    # --- 2. Create the graph for min-cut ---
    G = nx.DiGraph()
    source = "graph_source_node"  # Use strings to avoid collision with element nodes if they are integers
    sink = "graph_sink_node"
    G.add_node(source)
    G.add_node(sink)

    # Add nodes for each element in T (using their indices 0 to n-1 as node IDs)
    for i in range(n):
        G.add_node(i)

    # --- 3. Add edges for the modular part m(X) ---
    # MinCut corresponds to min(g(X) - K), where K = sum_{i:w(i)<0} w(i)
    K_offset = 0.0 # Constant offset used in the reduction
    for i in range(n):
        wi = w[i]
        if wi > 0:
            # Add edge (i, t) with capacity wi
            # Cost wi is added if node i is NOT in X (i.e., i is on source side S)
            G.add_edge(i, sink, capacity=wi)
        elif wi < 0:
            # Add edge (s, i) with capacity -wi
            # Cost -wi is added if node i IS in X (i.e., i is on sink side T)
            G.add_edge(source, i, capacity=-wi)
            K_offset += wi # K is sum of negative weights

    # --- 4. Add edges & auxiliary nodes for submodular part b(X) ---
    # This implements the reduction for sum_{j<k} f_jk(|X intersect S_jk|)
    # using the chain gadget based on Ishikawa (2003) / standard libraries.
    # Requires v_jk(i) >= v_jk(i+1) >= 0.
    aux_node_counter = 0 # To generate unique auxiliary node IDs

    for j_idx in range(n):
        for k_idx in range(j_idx + 1, n):
            # Define the interval S_jk based on indices j_idx to k_idx
            S_jk_indices = list(range(j_idx, k_idx + 1))
            p = len(S_jk_indices)
            if p == 0: continue

            interval_elements = [element_list[idx] for idx in S_jk_indices]
            j_el = element_list[j_idx]
            k_el = element_list[k_idx]

            # Add p auxiliary nodes for this interval: a_1, ..., a_p
            # Use unique IDs like "aux_j_k_r"
            current_aux_nodes = [f"aux_{j_idx}_{k_idx}_{r}" for r in range(1, p + 1)]
            for aux_node in current_aux_nodes:
                G.add_node(aux_node)

            # Add edges based on v_jk(i) = f_jk(i) - f_jk(i-1) >= 0
            try:
                # Edge (s, a_1) with capacity v_jk(1)
                v1 = v_func(j_el, k_el, 1)
                if v1 < 0: raise ValueError(f"v({j_el},{k_el},1) = {v1} must be non-negative.")
                if v1 > 0: G.add_edge(source, current_aux_nodes[0], capacity=v1)

                # Edges (a_r, a_{r+1}) with capacity v_jk(r+1) for r=1..p-1
                for r in range(p - 1):
                    v_r_plus_1 = v_func(j_el, k_el, r + 2) # Index i=r+2 for v_jk
                    if v_r_plus_1 < 0: raise ValueError(f"v({j_el},{k_el},{r+2}) = {v_r_plus_1} must be non-negative.")
                    # Check concavity: v(r+1) >= v(r+2)
                    v_r_plus_1_prev = v_func(j_el, k_el, r+1) # Need v(r+1) for check
                    if v_r_plus_1 > v_r_plus_1_prev + 1e-9: # Add tolerance for float comparison
                         raise ValueError(f"Concavity violated: v({j_el},{k_el},{r+1})={v_r_plus_1_prev} < v({j_el},{k_el},{r+2})={v_r_plus_1}")

                    if v_r_plus_1 > 0: G.add_edge(current_aux_nodes[r], current_aux_nodes[r+1], capacity=v_r_plus_1)

                # Edge (a_p, t) with capacity v_jk(p+1) (often 0)
                try:
                    v_p_plus_1 = v_func(j_el, k_el, p + 1)
                    if v_p_plus_1 < 0: raise ValueError(f"v({j_el},{k_el},{p+1}) = {v_p_plus_1} must be non-negative.")
                    # Check concavity: v(p) >= v(p+1)
                    v_p = v_func(j_el,k_el,p)
                    if v_p_plus_1 > v_p + 1e-9:
                        raise ValueError(f"Concavity violated: v({j_el},{k_el},{p})={v_p} < v({j_el},{k_el},{p+1})={v_p_plus_1}")

                    if v_p_plus_1 > 0: G.add_edge(current_aux_nodes[p-1], sink, capacity=v_p_plus_1)
                except IndexError:
                    pass # Assume v=0 if index p+1 is out of bounds for v_func definition

                # Edges from element nodes l_r to auxiliary nodes a_r (cap infinity)
                for r in range(p):
                    element_node_idx = S_jk_indices[r] # Node ID (0 to n-1) for element
                    aux_node_id = current_aux_nodes[r]
                    G.add_edge(element_node_idx, aux_node_id, capacity=float('inf'))

            except Exception as e:
                print(f"Error processing interval ({j_el}, {k_el}): {e}")
                raise # Re-raise after printing context


    # --- 5. Compute the max flow / min cut ---
    # The min_cut_value calculated by nx = min (g(X) - K_offset)
    try:
         min_cut_value, partition = nx.minimum_cut(G, source, sink)
    except nx.NetworkXUnbounded:
         # This can happen if there's a path from s to t with infinite capacity.
         # Should not happen with the construction if costs/values are finite. Check inf edges.
         raise RuntimeError("Min-cut is unbounded. Check graph construction, especially infinite capacity edges.")
    except Exception as e:
         print(f"Error during minimum_cut computation: {e}")
         # Consider trying a different algorithm if available e.g. preflow=True
         # min_cut_value, partition = nx.minimum_cut(G, source, sink, capacity='capacity', flow_func=nx.algorithms.flow.preflow_push)
         raise

    # The set X_min corresponds to element nodes on the sink side 'T' of the cut.
    # Networkx partition returns (S, T) where S is reachable from source.
    source_side_nodes, sink_side_nodes = partition

    X_min_indices = {node for node in sink_side_nodes if isinstance(node, int) and 0 <= node < n}
    X_min_set = {element_list[idx] for idx in X_min_indices}

    # --- 6. Calculate the minimum value of the original function f(X) ---
    # min g(X) = min_cut_value + K_offset
    min_g_X = min_cut_value + K_offset

    # Calculate constant term a_overline(A)
    a_overline_A_val = 0.0
    for element in A_set:
        cost = a_overline_func(element)
        if cost < 0: raise ValueError(f"a_overline({element}) = {cost} must be non-negative.")
        a_overline_A_val += cost

    # min f(X) = min g(X) + a_overline(A)
    min_f_X = min_g_X + a_overline_A_val

    return min_f_X, X_min_set

# --- Example Usage ---
if __name__ == '__main__':

    # --- Parameters ---
    T = list(range(5)) # Ground set T = {0, 1, 2, 3, 4}
    A = {1, 3}         # Fixed set A

    # Example v_jk(i) = max(0, 3 - i) * factor(j,k)
    # Let factor=1/(k-j+1) for variety
    def example_v_func(j, k, i):
        # i is 1-based index
        if i <= 0: return 0
        # factor = 1.0 / (element_to_idx[k] - element_to_idx[j] + 1) # Example factor
        factor = 1.0 # Simpler factor
        val = max(0.0, 3.0 - float(i)) * factor
        # Add slight noise to test concavity check robustness if needed
        # val -= np.random.rand() * 0.001
        # print(f"v({j},{k},{i}) = {val}") # Debug
        return val

    def example_a_underline(element):
        return 1.0 if element >= 3 else 0.0

    def example_a_overline(element):
        return 2.0 if element < 2 else 0.0

    # Map elements to indices for direct calculation if needed later
    element_to_idx = {el: i for i, el in enumerate(T)}

    # --- Solve ---
    try:
        min_val, min_set = solve_submodular_min_intervals(
            T, A, example_v_func, example_a_underline, example_a_overline
        )

        print(f"Ground Set T: {T}")
        print(f"Fixed Set A: {A}")
        print(f"Minimum value f(X): {min_val:.4f}")
        print(f"Minimizing set X: {min_set}")

    except Exception as e:
        print(f"An error occurred: {e}")
        import traceback
        traceback.print_exc()


    # --- Optional Verification (for small N) ---
    def calculate_f_direct(X_set, T_elements, A_set, v_func, a_underline_func, a_overline_func):
         # Direct calculation of f(X) = b(X) - a_underline(X \\ A) + a_overline(A \\ X)
         n_direct = len(T_elements)
         element_list_direct = list(T_elements)
         element_to_idx_direct = {el: i for i, el in enumerate(element_list_direct)}

         # Calculate b(X)
         b_X = 0.0
         for j_idx in range(n_direct):
             for k_idx in range(j_idx + 1, n_direct):
                 S_jk_indices = list(range(j_idx, k_idx + 1))
                 S_jk_elements = {element_list_direct[idx] for idx in S_jk_indices}

                 X_intersect_Sjk = X_set.intersection(S_jk_elements)
                 p = len(X_intersect_Sjk)

                 j_el = element_list_direct[j_idx]
                 k_el = element_list_direct[k_idx]

                 f_jk_p = 0.0
                 for i in range(1, p + 1):
                     try:
                         v = v_func(j_el, k_el, i)
                         if v < 0: print(f"Warning: Direct calc got neg v({j_el},{k_el},{i})={v}")
                         f_jk_p += max(0.0, v) # Ensure non-negative contribution if v_func was faulty
                     except IndexError: pass # v=0 beyond definition
                     except Exception as e_inner: print(f"Direct calc Error v({j_el},{k_el},{i}):{e_inner}")
                 b_X += f_jk_p

         # Calculate a_underline(X \ A)
         a_u_X_minus_A = sum(a_underline_func(el) for el in X_set if el not in A_set)

         # Calculate a_overline(A \ X)
         a_o_A_minus_X = sum(a_overline_func(el) for el in A_set if el not in X_set)

         f_X = b_X - a_u_X_minus_A + a_o_A_minus_X
         return f_X

    # You would typically only run verification if the main function succeeded
    if 'min_set' in locals():
        print("-" * 20)
        print("Verification (using direct calculation):")
        try:
            f_val_at_min_set = calculate_f_direct(min_set, T, A, example_v_func, example_a_underline, example_a_overline)
            print(f"f(X_min) calculated directly: {f_val_at_min_set:.4f}")
            # Check if the direct calculation matches the min-cut result
            if not np.isclose(min_val, f_val_at_min_set):
                 print("WARNING: Direct calculation differs from min-cut result!")

            # Example: Check f(empty_set)
            f_empty = calculate_f_direct(set(), T, A, example_v_func, example_a_underline, example_a_overline)
            print(f"f(emptyset) calculated directly: {f_empty:.4f}")

            # Example: Check f(T)
            f_T = calculate_f_direct(set(T), T, A, example_v_func, example_a_underline, example_a_overline)
            print(f"f(T) calculated directly: {f_T:.4f}")

        except Exception as e:
            print(f"Error during direct calculation verification: {e}")

Ground Set T: [0, 1, 2, 3, 4]
Fixed Set A: {1, 3}
Minimum value f(X): -1.0000
Minimizing set X: set()
--------------------
Verification (using direct calculation):
f(X_min) calculated directly: 2.0000
f(emptyset) calculated directly: 2.0000
f(T) calculated directly: 29.0000
